# Tick data

The same notebook as [`../tick_data.ipynb`](../tick_data.ipynb), written against
[`ib_async`](https://github.com/ib-api-reloaded/ib_async) instead of the TWS API
shape. The library is unmodified and installed as usual; `ibx.ib_async.attach`
replaces the one layer of it that expects a socket to a gateway.

Top of book as it changes, and every print as it happens.

## Connecting

`IB.connect` was written for a gateway, so it takes a host, a port and a client
id. Here it takes none of them: the credentials go to `attach`, and there is no
local process to reach.

`ib.sleep()` rather than `time.sleep()` throughout. The library's loop runs on
this thread, and a plain sleep stops it — every stream then reads as dead.

In [ ]:
import os
from dotenv import load_dotenv
from ib_async import IB, util
import ibx.ib_async

util.startLoop()
load_dotenv()

ib = ibx.ib_async.attach(
    IB(),
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)
ib.connect()          # names no host: there is no gateway to name

print(f"connected: {ib.isConnected()}")
print(f"accounts:  {ib.managedAccounts()}")

## Top of book

One `Ticker` per contract, amended in place. Reading it after a sleep reads
the latest the venue has sent.

In [ ]:
from ib_async import Stock

contracts = [Stock(s, "SMART", "USD") for s in ("AAPL", "MSFT", "SPY")]
ib.qualifyContracts(*contracts)

tickers = [ib.reqMktData(c) for c in contracts]
ib.sleep(4)

for t in tickers:
    print(f"{t.contract.symbol:6} bid {t.bid:>9}  ask {t.ask:>9}  last {t.last:>9}")

## Watching it change

`pendingTickers` is the library's own event: it fires with the tickers that
changed, rather than on a timer.

In [ ]:
seen = 0

def on_change(pending):
    global seen
    seen += len(pending)
    for t in pending:
        print(f"{t.contract.symbol:6} {t.time}  last {t.last}")

ib.pendingTickersEvent += on_change
ib.sleep(10)
ib.pendingTickersEvent -= on_change

print(f"\n{seen} updates in ten seconds")

In [ ]:
for c in contracts:
    ib.cancelMktData(c)

## Every print

Tick-by-tick is the trade stream itself rather than a summary of it.

In [ ]:
spy = contracts[-1]
ticker = ib.reqTickByTickData(spy, "Last")
ib.sleep(10)

print(f"{len(ticker.tickByTicks)} trades")
for t in ticker.tickByTicks[-5:]:
    print(f"{t.time}  {t.price:8.2f} x {t.size}")

ib.cancelTickByTickData(spy, "Last")

In [ ]:
ib.disconnect()